# Prerequisite Parsing
This notebook converts semi-structured course requirement text into structured prerequisite relationships that can be used for eligibility filtering in CalCourse.

In [2]:
import pandas as pd

courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

courses.shape

(2119, 8)

## 1. Requirement Patterns
Requirement text contains a mix of explicit course prereqs and non-course constraints such as instructor consent, standing, GPA requirements, and auditions.

In [3]:
requirements = courses[
    courses["requirements"].notna()
] [
    ["course_id", "subject", "course_number", "title", "requirements"]
].copy()

requirements.shape

(1157, 5)

In [4]:
requirements.sample(
    30, 
    random_state=42
)

,course_id,subject,course_number,title,requirements
1743,119627,PSYCH,130,Clinical Psychology,Recommended: Psychology 1 or Psychology 2
448,104585,CYPLAN,199,Special Study for Advanced Undergraduates,Consent of instructor.
202,163668,CMPBIO,C149,Computational Functional Genomics,MATH 54 or ELENG 64/ELENG 66; COMPSCI 61A or e...
504,105196,ECON,136,Financial Economics,"100A or 101A, and one semester of statistics."
1373,116762,MELC,R1A,Reading and Composition in Middle Eastern Lang...,Satisfaction of the Entry Level Writing Requir...
309,103334,CHMENG,162,Dynamics and Control of Chemical Processes,Chemical and Biomolecular Engineering 142 and ...
1594,125433,PHYSICS,5CL,Introduction to Experimental Physics II,Physics 5B & 5BL or 7B; Physics 5C or 7C (whic...
1052,110948,INTEGBI,112,Horticultural Methods in the Botanical Garden,Consent of instructor.
1747,119639,PSYCH,140,Developmental Psychology,Recommended: Psych 1
1125,111556,JAPAN,155,Modern Japanese Literature,Japanese 100A (may be taken concurrently).


### Requirement pattern observations
- Requirement text is not consistently structured. 
- Some courses like explicit prereqs such as 'MATH 53' or 'COMPSCI 61A'.
- Other requirements include instructor consent, class standing, GPA thresholds, auditions, or equivalent-background language. 
- Because these constraints are mixed together, CalCourse V1 will first focus on extracting recognizable course references rather than fully interpreting every eligibility rule. 

## 2. Explicit Course Extraction
CalCourse extracts course references from requirement text using regular expression, then validates each match against known Berkeley subject codes. 
This helps remove false positives such as 'OR', 'AND', and 'TO', which can otherwise look like department names. 

In [5]:
import re

In [6]:
valid_subjects = set(courses["subject"].str.upper())

In [7]:
course_pattern = r"\b([A-Z]+)\s+([A-Z]?\d+[A-Z]*)\b"

In [8]:
def extract_courses(text):
    matches = re.findall(course_pattern, text.upper())

    return [
        (subject, number)
        for subject, number in matches
        if subject in valid_subjects
    ]

In [9]:
example = requirements.iloc[0]["requirements"]

print(example)
print(re.findall(course_pattern, example.upper()))

Prerequisite: MATH 51, MATH 52, MATH 53 (may be taken concurrently), PHYSICS 7A; and programming (COMPSCI 61A or ENGIN 7).
[('MATH', '51'), ('MATH', '52'), ('MATH', '53'), ('PHYSICS', '7A'), ('COMPSCI', '61A'), ('ENGIN', '7')]


In [10]:
sample = requirements.sample(20, random_state=42).copy()

sample["parsed_courses"] = sample["requirements"].apply(extract_courses)

sample[
    ["subject", "course_number", "requirements", "parsed_courses"]
]

,subject,course_number,requirements,parsed_courses
1743,PSYCH,130,Recommended: Psychology 1 or Psychology 2,[]
448,CYPLAN,199,Consent of instructor.,[]
202,CMPBIO,C149,MATH 54 or ELENG 64/ELENG 66; COMPSCI 61A or e...,"[(MATH, 54), (ELENG, 64), (ELENG, 66), (COMPSC..."
504,ECON,136,"100A or 101A, and one semester of statistics.",[]
1373,MELC,R1A,Satisfaction of the Entry Level Writing Requir...,[]
309,CHMENG,162,Chemical and Biomolecular Engineering 142 and ...,[]
1594,PHYSICS,5CL,Physics 5B & 5BL or 7B; Physics 5C or 7C (whic...,"[(PHYSICS, 5B), (PHYSICS, 5C)]"
1052,INTEGBI,112,Consent of instructor.,[]
1747,PSYCH,140,Recommended: Psych 1,"[(PSYCH, 1)]"
1125,JAPAN,155,Japanese 100A (may be taken concurrently).,[]


### Extraction observations 
- Validating matches against known Berkeley subject codes removes many obvious false positives. 
- Explicit references such as MATH54, ELENG64, and PHYSICS5B can be extracted reliably. 
- Some requirements use department names rather than subject codes, or omit the subject when listing additional courses, so those cases require additional normalization. 
- Non-course constraints such as consent, standing, and auditions correctly produce no course matches.

## 3. Subject Name Normalization
The goal here is to catch cases where the requirement text says 'Psychology 1' instead of the canonical subject code 'PSYCH 1', or 'Japanese 100A' instead of 'JAPAN 100A'.

In [11]:
subject_aliases = {
    "PSYCHOLOGY": "PSYCH",
    "JAPANESE": "JAPAN",
    "ECONOMICS": "ECON",
}

In [12]:
def normalize_subject_names(text): 
    normalized = text.upper()

    for alias, subject in subject_aliases.items():
        normalized = normalized.replace(alias, subject)

    return normalized

In [23]:
# update extractor

def extract_courses(text):
    text = normalize_subject_names(text)

    matches = re.findall(course_pattern, text)

    return [
        (subject, number) 
        for subject, number in matches
        if subject in valid_subjects
    ]

In [24]:
sample["parsed_courses"] = sample["requirements"].apply(extract_courses)

sample[
    ["subject", "course_number", "requirements", "parsed_courses"]
]

,subject,course_number,requirements,parsed_courses
1743,PSYCH,130,Recommended: Psychology 1 or Psychology 2,"[(PSYCH, 1), (PSYCH, 2)]"
448,CYPLAN,199,Consent of instructor.,[]
202,CMPBIO,C149,MATH 54 or ELENG 64/ELENG 66; COMPSCI 61A or e...,"[(MATH, 54), (ELENG, 64), (ELENG, 66), (COMPSC..."
504,ECON,136,"100A or 101A, and one semester of statistics.",[]
1373,MELC,R1A,Satisfaction of the Entry Level Writing Requir...,[]
309,CHMENG,162,Chemical and Biomolecular Engineering 142 and ...,[]
1594,PHYSICS,5CL,Physics 5B & 5BL or 7B; Physics 5C or 7C (whic...,"[(PHYSICS, 5B), (PHYSICS, 5C)]"
1052,INTEGBI,112,Consent of instructor.,[]
1747,PSYCH,140,Recommended: Psych 1,"[(PSYCH, 1)]"
1125,JAPAN,155,Japanese 100A (may be taken concurrently).,"[(JAPAN, 100A)]"


## 4. Shorthard Course References
Some requirement text lists the subject once and then omits it for additional courses, such as 'MATH 54 or 55'. This section adds simple logic so later course numbers can inherit the most recent valid subject.

In [25]:
def extract_courses_with_shorthand(text):
    text = normalize_subject_names(text)

    tokens = re.findall(
        r"\b([A-Z]+)?\s*([A-Z]?\d+[A-Z]*)\b",
        text
    )

    extracted = []
    current_subject = None
    connectors = {"", "OR", "AND"}

    for subject, number in tokens:

        if subject in valid_subjects:
            current_subject = subject
            extracted.append((subject, number))

        elif subject in connectors and current_subject is not None:
            extracted.append((current_subject, number))

    return extracted

In [26]:
test_text = "MATH 54 or 55"

print(extract_courses_with_shorthand(test_text))

[('MATH', '54'), ('MATH', '55')]


In [27]:
tests = [
    "MATH 54 or 55",
    "PHYSICS 5B or 7B",
    "MATH 51; MATH 53; and 54",
    "PSYCHOLOGY 1 or Psychology 2"
]

for text in tests:
    print(text)
    print(extract_courses_with_shorthand(text))
    print()

MATH 54 or 55
[('MATH', '54'), ('MATH', '55')]

PHYSICS 5B or 7B
[('PHYSICS', '5B'), ('PHYSICS', '7B')]

MATH 51; MATH 53; and 54
[('MATH', '51'), ('MATH', '53'), ('MATH', '54')]

PSYCHOLOGY 1 or Psychology 2
[('PSYCH', '1'), ('PSYCH', '2')]



In [28]:
sample["parsed_courses_v2"] = sample["requirements"].apply(
    extract_courses_with_shorthand
)

sample[
    [
        "subject",
        "course_number",
        "requirements",
        "parsed_courses",
        "parsed_courses_v2"
    ]
]

,subject,course_number,requirements,parsed_courses,parsed_courses_v2
1743,PSYCH,130,Recommended: Psychology 1 or Psychology 2,"[(PSYCH, 1), (PSYCH, 2)]","[(PSYCH, 1), (PSYCH, 2)]"
448,CYPLAN,199,Consent of instructor.,[],[]
202,CMPBIO,C149,MATH 54 or ELENG 64/ELENG 66; COMPSCI 61A or e...,"[(MATH, 54), (ELENG, 64), (ELENG, 66), (COMPSC...","[(MATH, 54), (ELENG, 64), (ELENG, 66), (COMPSC..."
504,ECON,136,"100A or 101A, and one semester of statistics.",[],[]
1373,MELC,R1A,Satisfaction of the Entry Level Writing Requir...,[],[]
309,CHMENG,162,Chemical and Biomolecular Engineering 142 and ...,[],[]
1594,PHYSICS,5CL,Physics 5B & 5BL or 7B; Physics 5C or 7C (whic...,"[(PHYSICS, 5B), (PHYSICS, 5C)]","[(PHYSICS, 5B), (PHYSICS, 5BL), (PHYSICS, 7B),..."
1052,INTEGBI,112,Consent of instructor.,[],[]
1747,PSYCH,140,Recommended: Psych 1,"[(PSYCH, 1)]","[(PSYCH, 1)]"
1125,JAPAN,155,Japanese 100A (may be taken concurrently).,"[(JAPAN, 100A)]","[(JAPAN, 100A)]"


## 5. Parser Validation
The updated parser is tested on a sample of requirement strings to check whether shorthand handling improves extraction without introducing pobvious false positives.

In [29]:
validation_sample = requirements.sample(
    30,
    random_state=7
).copy()

validation_sample["parsed_v1"] = validation_sample["requirements"].apply(
    extract_courses
)

validation_sample["parsed_v2"] = validation_sample["requirements"].apply(
    extract_courses_with_shorthand
)

validation_sample[
    [
        "subject",
        "course_number",
        "requirements",
        "parsed_v1",
        "parsed_v2"
    ]
]

,subject,course_number,requirements,parsed_v1,parsed_v2
1233,MATH,124,"Math 53, 54, 55","[(MATH, 53)]","[(MATH, 53), (MATH, 54), (MATH, 55)]"
1263,MATH,56,Prerequisites are Math 52 (previously known as...,"[(MATH, 52), (MATH, 1B)]","[(MATH, 52), (MATH, 1B), (MATH, N1B), (MATH, 1..."
2000,UGBA,160,106.,[],[]
1118,ITALIAN,H195,"3.3 overall GPA, 3.5 GPA in the major and must...",[],[]
202,CMPBIO,C149,MATH 54 or ELENG 64/ELENG 66; COMPSCI 61A or e...,"[(MATH, 54), (ELENG, 64), (ELENG, 66), (COMPSC...","[(MATH, 54), (ELENG, 64), (ELENG, 66), (COMPSC..."
860,FRENCH,H195A,Open to seniors majoring in French who meet th...,[],[]
1405,MUSIC,150,"Music 52A, Music 90, or consent of instructor.","[(MUSIC, 52A), (MUSIC, 90)]","[(MUSIC, 52A), (MUSIC, 90)]"
236,CHEM,12A,12A: 1B or 4B with grade of C- or higher. For ...,[],[]
2111,XRHETOR,R1A,UC Entry Level Writing Requirement or UC Analy...,[],[]
301,CHINESE,4B,CHINESE 4A,"[(CHINESE, 4A)]","[(CHINESE, 4A)]"


In [30]:
changed = validation_sample[
    validation_sample["parsed_v1"].astype(str)
    != validation_sample["parsed_v2"].astype(str)
]

changed[
    [
        "subject",
        "course_number",
        "requirements",
        "parsed_v1",
        "parsed_v2"
    ]
]

,subject,course_number,requirements,parsed_v1,parsed_v2
1233,MATH,124,"Math 53, 54, 55","[(MATH, 53)]","[(MATH, 53), (MATH, 54), (MATH, 55)]"
1263,MATH,56,Prerequisites are Math 52 (previously known as...,"[(MATH, 52), (MATH, 1B)]","[(MATH, 52), (MATH, 1B), (MATH, N1B), (MATH, 1..."
1576,PHYSICS,105,"Physics 5A, 5B, 5C or 7A, 7B, 7C","[(PHYSICS, 5A)]","[(PHYSICS, 5A), (PHYSICS, 5B), (PHYSICS, 5C), ..."


### Validation observations 
- The shorthand-aware parser improves the extraction on requirements where the subject is only written once. 
- Most requirements strings remain unchanged, suggesting the new logic is targeted rather than overly aggressive. 
- The parser still doesn't attempt to interpret non-course constraints such as GPA, standing, consent, or auditions.

## 6. Build Structured Prerequisite Table 
The validated parser is applied to all requirement text, and extracted course references are expanded into a row-level prerequisite table for downstream eligibility filtering. 

In [31]:
requirements["parsed_courses"] = requirements["requirements"].apply(
    extract_courses_with_shorthand
)

In [32]:
prereq_table = requirements[
    ["course_id", "subject", "course_number", "title", "parsed_courses"]
].explode("parsed_courses")

In [33]:
prereq_table = prereq_table[
    prereq_table["parsed_courses"].notna()
].copy()

In [34]:
prereq_table[["prereq_subject", "prereq_number"]] = pd.DataFrame(
    prereq_table["parsed_courses"].tolist(),
    index=prereq_table.index
)

In [35]:
prereq_table = prereq_table[
    [
        "course_id",
        "subject",
        "course_number",
        "prereq_subject",
        "prereq_number"
    ]
]

In [36]:
prereq_table.head(20)

,course_id,subject,course_number,prereq_subject,prereq_number
1,162181,AEROENG,10,MATH,51
1,162181,AEROENG,10,MATH,52
1,162181,AEROENG,10,MATH,53
1,162181,AEROENG,10,PHYSICS,7A
1,162181,AEROENG,10,COMPSCI,61A
1,162181,AEROENG,10,ENGIN,7
2,166420,AEROENG,100,MECENG,103
2,166420,AEROENG,100,MECENG,104
2,166420,AEROENG,100,MECENG,132
2,166420,AEROENG,100,MECENG,106


In [37]:
prereq_table.shape

(1068, 5)

In [38]:
prereq_table.duplicated().sum()

np.int64(40)

In [39]:
prereq_table = prereq_table.drop_duplicates()

In [40]:
prereq_table["course_id"].nunique()

400

In [41]:
prereq_table.to_csv(
    "../data/processed/prerequisites_fall_2026.csv",
    index=False
)

## Structured Prerequisite Output
The parser produced 1,068 course-to-prerequisite relationships in a normalized table. Each row represents a directed prerequisite link that can be used for eligibility filtering an prerequisite graph construction.